In [ ]:
# Installation commands (run these first in your environment)
install_commands = """
# Core scientific computing
pip install numpy pandas scipy scikit-learn matplotlib seaborn --break-system-packages

# Chemistry and molecular modeling
pip install rdkit biopython openbabel-wheel --break-system-packages

# Machine learning and statistics
pip install xgboost lightgbm catboost statsmodels --break-system-packages

# Molecular docking and analysis
pip install vina autodock-vina meeko --break-system-packages

# Data handling
pip install requests tqdm joblib --break-system-packages

# Visualization
pip install plotly kaleido --break-system-packages

# Statistical analysis
pip install pingouin --break-system-packages
"""

print(install_commands)
print("\n" + "="*80)
print("Please run the above pip install commands before proceeding to Part 2")
print("="*80)


# Core scientific computing
pip install numpy pandas scipy scikit-learn matplotlib seaborn --break-system-packages

# Chemistry and molecular modeling
pip install rdkit biopython openbabel-wheel --break-system-packages

# Machine learning and statistics
pip install xgboost lightgbm catboost statsmodels --break-system-packages

# Molecular docking and analysis
pip install vina autodock-vina meeko --break-system-packages

# Data handling
pip install requests tqdm joblib --break-system-packages

# Visualization
pip install plotly kaleido --break-system-packages

# Statistical analysis
pip install pingouin --break-system-packages


Please run the above pip install commands before proceeding to Part 2


In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.6/36.6 MB 45.6 MB/s eta 0:00:00


In [ ]:
"""
COMPREHENSIVE DRUG DISCOVERY PIPELINE FOR TARGET PROTEIN 3NVW
Part 2: Library Imports and Configuration
===============================================================================
"""

import os
import sys
import warnings
import pickle
import gzip
import zipfile
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass

# Core scientific computing
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import cdist

# Chemistry libraries
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Lipinski, Crippen, MolSurf
from rdkit.Chem import PandasTools, Fragments, rdMolDescriptors
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit.Chem import Draw
from rdkit import DataStructs

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve, f1_score,
    matthews_corrcoef
)
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

# Additional ML models
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("Warning: XGBoost not available")

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Utilities
import requests
from tqdm import tqdm
import joblib

# BioPython for protein handling
try:
    from Bio.PDB import PDBParser, PDBIO, Select
    BIOPYTHON_AVAILABLE = True
except ImportError:
    BIOPYTHON_AVAILABLE = False
    print("Warning: BioPython not available - will download PDB directly")

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

@dataclass
class Config:
    """Configuration parameters for the drug discovery pipeline"""

    # Target protein
    TARGET_PDB_ID: str = "3nvw"
    TARGET_PDB_URL: str = "https://files.rcsb.org/download/3nvw.pdb"

    # COCONUT Database
    COCONUT_URL: str = "https://coconut.s3.uni-jena.de/prod/downloads/2026-01/coconut_sdf_3d-01-2026.zip"
    COCONUT_CSV: str = "COCONUT_DB.csv"

    # Screening parameters
    MAX_COMPOUNDS: int = 100000  # Maximum compounds to screen (adjust based on resources)
    BATCH_SIZE: int = 10000
    TOP_N_LIGANDS: int = 50

    # ML Model parameters
    TARGET_ACCURACY: float = 0.90  # Lowered from 0.95 as requested
    MIN_TRAINING_SIZE: int = 5000  # Increased training data
    RANDOM_STATE: int = 42
    N_ESTIMATORS: int = 200

    # Molecular property filters (Lipinski's Rule of Five + Veber's rules)
    MW_MIN: float = 150
    MW_MAX: float = 500
    LOGP_MIN: float = -0.5
    LOGP_MAX: float = 5.0
    HBA_MAX: int = 10
    HBD_MAX: int = 5
    TPSA_MAX: float = 140
    ROTATABLE_BONDS_MAX: int = 10

    # Docking parameters
    ENERGY_THRESHOLD: float = -6.0  # Increased threshold as requested

    # Output directories
    OUTPUT_DIR: str = "screening_results_3nvw"
    FIGURES_DIR: str = "figures"
    MODELS_DIR: str = "models"
    DATA_DIR: str = "data"

    def __post_init__(self):
        """Create output directories"""
        for dir_path in [self.OUTPUT_DIR,
                        f"{self.OUTPUT_DIR}/{self.FIGURES_DIR}",
                        f"{self.OUTPUT_DIR}/{self.MODELS_DIR}",
                        f"{self.OUTPUT_DIR}/{self.DATA_DIR}"]:
            Path(dir_path).mkdir(parents=True, exist_ok=True)

# Initialize configuration
config = Config()

print("="*80)
print("DRUG DISCOVERY PIPELINE CONFIGURATION")
print("="*80)
print(f"Target Protein: {config.TARGET_PDB_ID}")
print(f"Database: COCONUT Natural Products")
print(f"Maximum Compounds to Screen: {config.MAX_COMPOUNDS:,}")
print(f"Target Accuracy: {config.TARGET_ACCURACY}")
print(f"Top Ligands to Select: {config.TOP_N_LIGANDS}")
print(f"Energy Threshold: {config.ENERGY_THRESHOLD} kcal/mol")
print(f"Output Directory: {config.OUTPUT_DIR}")
print("="*80)

DRUG DISCOVERY PIPELINE CONFIGURATION
Target Protein: 3nvw
Database: COCONUT Natural Products
Maximum Compounds to Screen: 100,000
Target Accuracy: 0.9
Top Ligands to Select: 50
Energy Threshold: -6.0 kcal/mol
Output Directory: screening_results_3nvw


In [ ]:
"""
COMPREHENSIVE DRUG DISCOVERY PIPELINE FOR TARGET PROTEIN 3NVW
Part 3: Molecular Descriptor Calculation (QSAR Features)
===============================================================================
"""

class MolecularDescriptors:
    """
    Comprehensive molecular descriptor calculator for QSAR modeling
    Includes physicochemical, topological, and fingerprint-based descriptors
    """

    @staticmethod
    def calculate_lipinski_descriptors(mol) -> Dict[str, float]:
        """Calculate Lipinski's Rule of Five descriptors"""
        try:
            return {
                'MW': Descriptors.MolWt(mol),
                'LogP': Descriptors.MolLogP(mol),
                'HBA': Descriptors.NumHAcceptors(mol),
                'HBD': Descriptors.NumHDonors(mol),
                'RotatableBonds': Descriptors.NumRotatableBonds(mol),
                'TPSA': Descriptors.TPSA(mol)
            }
        except:
            return None

    @staticmethod
    def calculate_advanced_descriptors(mol) -> Dict[str, float]:
        """Calculate advanced molecular descriptors for QSAR"""
        try:
            descriptors = {
                # Topological descriptors
                'BertzCT': Descriptors.BertzCT(mol),
                'Chi0': Descriptors.Chi0(mol),
                'Chi1': Descriptors.Chi1(mol),
                'HallKierAlpha': Descriptors.HallKierAlpha(mol),
                'Kappa1': Descriptors.Kappa1(mol),
                'Kappa2': Descriptors.Kappa2(mol),
                'Kappa3': Descriptors.Kappa3(mol),

                # Electronic descriptors
                'NumValenceElectrons': Descriptors.NumValenceElectrons(mol),
                'NumRadicalElectrons': Descriptors.NumRadicalElectrons(mol),

                # Structural descriptors
                'RingCount': Descriptors.RingCount(mol),
                'AromaticRings': Descriptors.NumAromaticRings(mol),
                'SaturatedRings': Descriptors.NumSaturatedRings(mol),
                'AliphaticRings': Descriptors.NumAliphaticRings(mol),
                'HeteroatomCount': Descriptors.NumHeteroatoms(mol),
                'HeavyAtomCount': Descriptors.HeavyAtomCount(mol),

                # Charge descriptors
                'FormalCharge': Chem.GetFormalCharge(mol),
                'NumAtoms': mol.GetNumAtoms(),

                # Surface area and volume
                'LabuteASA': Descriptors.LabuteASA(mol),
                'PEOE_VSA1': Descriptors.PEOE_VSA1(mol),
                'PEOE_VSA2': Descriptors.PEOE_VSA2(mol),
                'SMR_VSA1': Descriptors.SMR_VSA1(mol),
                'SlogP_VSA1': Descriptors.SlogP_VSA1(mol),

                # Fragment counts
                'fr_Al_COO': Fragments.fr_Al_COO(mol),
                'fr_Al_OH': Fragments.fr_Al_OH(mol),
                'fr_Ar_N': Fragments.fr_Ar_N(mol),
                'fr_Ar_OH': Fragments.fr_Ar_OH(mol),
                'fr_COO': Fragments.fr_COO(mol),
                'fr_COO2': Fragments.fr_COO2(mol),
                'fr_C_O': Fragments.fr_C_O(mol),
                'fr_NH0': Fragments.fr_NH0(mol),
                'fr_NH1': Fragments.fr_NH1(mol),
                'fr_NH2': Fragments.fr_NH2(mol),
                'fr_Ndealkylation1': Fragments.fr_Ndealkylation1(mol),
                'fr_Nhpyrrole': Fragments.fr_Nhpyrrole(mol),
                'fr_SH': Fragments.fr_SH(mol),
                'fr_aldehyde': Fragments.fr_aldehyde(mol),
                'fr_benzene': Fragments.fr_benzene(mol),
                'fr_furan': Fragments.fr_furan(mol),
                'fr_imidazole': Fragments.fr_imidazole(mol),
                'fr_ketone': Fragments.fr_ketone(mol),
                'fr_morpholine': Fragments.fr_morpholine(mol),
                'fr_phenol': Fragments.fr_phenol(mol),
                'fr_piperdine': Fragments.fr_piperdine(mol),
                'fr_pyridine': Fragments.fr_pyridine(mol),

                # Additional important descriptors
                'FractionCsp3': Descriptors.FractionCSP3(mol),
                'NumSaturatedCarbocycles': Descriptors.NumSaturatedCarbocycles(mol),
                'NumAromaticCarbocycles': Descriptors.NumAromaticCarbocycles(mol),
                'NumSaturatedHeterocycles': Descriptors.NumSaturatedHeterocycles(mol),
                'NumAromaticHeterocycles': Descriptors.NumAromaticHeterocycles(mol),
            }

            return descriptors
        except Exception as e:
            print(f"Error calculating advanced descriptors: {e}")
            return None

    @staticmethod
    def calculate_fingerprint(mol, radius=2, n_bits=2048) -> np.ndarray:
        """Calculate Morgan fingerprint"""
        try:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            return np.array(fp)
        except:
            return None

    @staticmethod
    def calculate_all_descriptors(mol, include_fingerprint=False) -> Dict[str, float]:
        """Calculate all molecular descriptors"""
        if mol is None:
            return None

        # Combine all descriptors
        lipinski = MolecularDescriptors.calculate_lipinski_descriptors(mol)
        advanced = MolecularDescriptors.calculate_advanced_descriptors(mol)

        if lipinski is None or advanced is None:
            return None

        all_descriptors = {**lipinski, **advanced}

        if include_fingerprint:
            fp = MolecularDescriptors.calculate_fingerprint(mol)
            if fp is not None:
                # Add fingerprint bits as features
                for i, bit in enumerate(fp):
                    all_descriptors[f'FP_{i}'] = int(bit)

        return all_descriptors


class DrugLikenessFilter:
    """
    Filter molecules based on drug-likeness criteria
    Implements Lipinski's Rule of Five and additional ADME filters
    """

    def __init__(self, config):
        self.config = config

    def passes_lipinski(self, mol) -> bool:
        """Check if molecule passes Lipinski's Rule of Five"""
        try:
            mw = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            hbd = Descriptors.NumHDonors(mol)
            hba = Descriptors.NumHAcceptors(mol)

            return (
                self.config.MW_MIN <= mw <= self.config.MW_MAX and
                self.config.LOGP_MIN <= logp <= self.config.LOGP_MAX and
                hbd <= self.config.HBD_MAX and
                hba <= self.config.HBA_MAX
            )
        except:
            return False

    def passes_veber(self, mol) -> bool:
        """Check if molecule passes Veber's rules"""
        try:
            tpsa = Descriptors.TPSA(mol)
            rotatable = Descriptors.NumRotatableBonds(mol)

            return (
                tpsa <= self.config.TPSA_MAX and
                rotatable <= self.config.ROTATABLE_BONDS_MAX
            )
        except:
            return False

    def passes_pains_filter(self, mol) -> bool:
        """Check if molecule is free from PAINS (Pan Assay Interference Compounds)"""
        try:
            params = FilterCatalogParams()
            params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
            catalog = FilterCatalog(params)
            return not catalog.HasMatch(mol)
        except:
            return True  # If filter fails, allow the molecule

    def is_druglike(self, mol) -> bool:
        """Combined drug-likeness filter"""
        return (
            self.passes_lipinski(mol) and
            self.passes_veber(mol) and
            self.passes_pains_filter(mol)
        )


def calculate_descriptors_for_dataframe(df: pd.DataFrame,
                                       smiles_col: str = 'smiles',
                                       include_fingerprint: bool = False) -> pd.DataFrame:
    """
    Calculate molecular descriptors for a DataFrame of SMILES

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing SMILES strings
    smiles_col : str
        Name of the column containing SMILES
    include_fingerprint : bool
        Whether to include fingerprint features

    Returns:
    --------
    pd.DataFrame with calculated descriptors
    """
    descriptor_calculator = MolecularDescriptors()
    results = []

    print(f"Calculating descriptors for {len(df)} molecules...")

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        smiles = row[smiles_col]
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            continue

        descriptors = descriptor_calculator.calculate_all_descriptors(
            mol, include_fingerprint=include_fingerprint
        )

        if descriptors is not None:
            descriptors['original_index'] = idx
            for col in df.columns:
                if col != smiles_col:
                    descriptors[col] = row[col]
            results.append(descriptors)

    if len(results) == 0:
        return pd.DataFrame()

    result_df = pd.DataFrame(results)
    print(f"Successfully calculated descriptors for {len(result_df)} molecules")

    return result_df


print("="*80)
print("Molecular Descriptor Calculator and Drug-Likeness Filters Loaded")
print("="*80)
print("Available descriptor sets:")
print("  1. Lipinski descriptors (MW, LogP, HBA, HBD, etc.)")
print("  2. Advanced QSAR descriptors (50+ features)")
print("  3. Morgan fingerprints (2048 bits)")
print("  4. Drug-likeness filters (Lipinski, Veber, PAINS)")
print("="*80)

Molecular Descriptor Calculator and Drug-Likeness Filters Loaded
Available descriptor sets:
  1. Lipinski descriptors (MW, LogP, HBA, HBD, etc.)
  2. Advanced QSAR descriptors (50+ features)
  3. Morgan fingerprints (2048 bits)
  4. Drug-likeness filters (Lipinski, Veber, PAINS)


In [ ]:
import os
import sys
import warnings
import pickle
import gzip
import zipfile
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass
import random # Import random module

# Core scientific computing
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import cdist

# Chemistry libraries
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Lipinski, Crippen, MolSurf
from rdkit.Chem import PandasTools, Fragments, rdMolDescriptors
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit.Chem import Draw
from rdkit import DataStructs

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve, f1_score,
    matthews_corrcoef
)
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

# Additional ML models
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("Warning: XGBoost not available")

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Utilities
import requests
from tqdm import tqdm
import joblib

# BioPython for protein handling
try:
    from Bio.PDB import PDBParser, PDBIO, Select
    BIOPYTHON_AVAILABLE = True
except ImportError:
    BIOPYTHON_AVAILABLE = False
    print("Warning: BioPython not available - will download PDB directly")

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

@dataclass
class Config:
    """Configuration parameters for the drug discovery pipeline"""

    # Target protein
    TARGET_PDB_ID: str = "3nvw"
    TARGET_PDB_URL: str = "https://files.rcsb.org/download/3nvw.pdb"

    # COCONUT Database
    COCONUT_URL: str = "https://coconut.s3.uni-jena.de/prod/downloads/2026-01/coconut_sdf_3d-01-2026.zip"
    COCONUT_CSV: str = "COCONUT_DB.csv"

    # Screening parameters
    MAX_COMPOUNDS: int = 100000  # Maximum compounds to screen (adjust based on resources)
    BATCH_SIZE: int = 10000
    TOP_N_LIGANDS: int = 50

    # ML Model parameters
    TARGET_ACCURACY: float = 0.90  # Lowered from 0.95 as requested
    MIN_TRAINING_SIZE: int = 5000  # Increased training data
    RANDOM_STATE: int = 42
    N_ESTIMATORS: int = 200

    # Molecular property filters (Lipinski's Rule of Five + Veber's rules)
    MW_MIN: float = 150
    MW_MAX: float = 500
    LOGP_MIN: float = -0.5
    LOGP_MAX: float = 5.0
    HBA_MAX: int = 10
    HBD_MAX: int = 5
    TPSA_MAX: float = 140
    ROTATABLE_BONDS_MAX: int = 10

    # Docking parameters
    ENERGY_THRESHOLD: float = -6.0  # Increased threshold as requested

    # Output directories
    OUTPUT_DIR: str = "screening_results_3nvw"
    FIGURES_DIR: str = "figures"
    MODELS_DIR: str = "models"
    DATA_DIR: str = "data"

    def __post_init__(self):
        """Create output directories"""
        for dir_path in [self.OUTPUT_DIR,
                        f"{self.OUTPUT_DIR}/{self.FIGURES_DIR}",
                        f"{self.OUTPUT_DIR}/{self.MODELS_DIR}",
                        f"{self.OUTPUT_DIR}/{self.DATA_DIR}"]:
            Path(dir_path).mkdir(parents=True, exist_ok=True)

# Initialize configuration
config = Config()

print("="*80)
print("DRUG DISCOVERY PIPELINE CONFIGURATION")
print("="*80)
print(f"Target Protein: {config.TARGET_PDB_ID}")
print(f"Database: COCONUT Natural Products")
print(f"Maximum Compounds to Screen: {config.MAX_COMPOUNDS:,}")
print(f"Target Accuracy: {config.TARGET_ACCURACY}")
print(f"Top Ligands to Select: {config.TOP_N_LIGANDS}")
print(f"Energy Threshold: {config.ENERGY_THRESHOLD} kcal/mol")
print(f"Output Directory: {config.OUTPUT_DIR}")
print("="*80)


class ProteinDataFetcher:
    """Fetch and prepare target protein structure"""

    def __init__(self, config):
        self.config = config

    def download_pdb_file(self, pdb_id: str = None) -> str:
        """
        Download PDB file from RCSB

        Parameters:
        -----------
        pdb_id : str
            PDB ID (default: from config)

        Returns:
        --------
        str : path to downloaded PDB file
        """
        if pdb_id is None:
            pdb_id = self.config.TARGET_PDB_ID

        pdb_file = f"{self.config.OUTPUT_DIR}/{self.config.DATA_DIR}/{pdb_id}.pdb"

        if os.path.exists(pdb_file):
            print(f"PDB file {pdb_id} already exists")
            return pdb_file

        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        print(f"Downloading PDB {pdb_id} from {url}...")

        try:
            response = requests.get(url)
            response.raise_for_status()

            with open(pdb_file, 'w') as f:
                f.write(response.text)

            print(f"Successfully downloaded {pdb_id}.pdb")
            return pdb_file

        except Exception as e:
            print(f"Error downloading PDB file: {e}")
            return None

    def prepare_protein_for_docking(self, pdb_file: str) -> str:
        """
        Prepare protein structure for docking (remove water, add hydrogens)

        Parameters:
        -----------
        pdb_file : str
            Path to PDB file

        Returns:
        --------
        str : path to prepared PDB file
        """
        print(f"Preparing protein structure for docking...")

        prepared_file = pdb_file.replace('.pdb', '_prepared.pdb')

        try:
            # Read PDB file
            with open(pdb_file, 'r') as f:
                lines = f.readlines()

            # Remove water molecules and other heteroatoms except ligands
            prepared_lines = []
            for line in lines:
                if line.startswith('ATOM') or line.startswith('HETATM'):
                    # Keep protein atoms, remove water
                    if 'HOH' not in line and 'WAT' not in line:
                        prepared_lines.append(line)
                elif line.startswith('CONECT'):
                    continue  # Skip connectivity records
                else:
                    prepared_lines.append(line)

            # Write prepared file
            with open(prepared_file, 'w') as f:
                f.writelines(prepared_lines)

            print(f"Protein prepared: {prepared_file}")
            return prepared_file

        except Exception as e:
            print(f"Error preparing protein: {e}")
            return pdb_file


class COCONUTDataLoader:
    """Load and preprocess COCONUT database"""

    def __init__(self, config):
        self.config = config

    def download_coconut_database(self) -> str:
        """
        Download COCONUT database from official source

        Returns:
        --------
        str : path to downloaded file
        """
        output_file = f"{self.config.OUTPUT_DIR}/{self.config.DATA_DIR}/coconut_3d.zip"

        if os.path.exists(output_file):
            print("COCONUT database already downloaded")
            return output_file

        url = self.config.COCONUT_URL
        print(f"Downloading COCONUT database from {url}...")
        print("This may take several minutes (file size: ~350 MB)...")

        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()

            total_size = int(response.headers.get('content-length', 0))

            with open(output_file, 'wb') as f:
                with tqdm(total=total_size, unit='B', unit_scale=True) as pbar:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))

            print(f"Successfully downloaded COCONUT database: {output_file}")
            return output_file

        except Exception as e:
            print(f"Error downloading COCONUT database: {e}")
            print("Please manually download from:")
            print(url)
            return None

    def extract_and_convert_to_csv(self, zip_file: str) -> str:
        """
        Extract SDF from zip and convert to CSV format

        Parameters:
        -----------
        zip_file : str
            Path to zip file

        Returns:
        --------
        str : path to CSV file
        """
        csv_file = f"{self.config.OUTPUT_DIR}/{self.config.DATA_DIR}/{self.config.COCONUT_CSV}"

        if os.path.exists(csv_file):
            print(f"CSV file already exists: {csv_file}")
            return csv_file

        print("Extracting and converting SDF to CSV...")

        try:
            # Extract zip file
            extract_dir = f"{self.config.OUTPUT_DIR}/{self.config.DATA_DIR}/coconut_extracted"
            Path(extract_dir).mkdir(exist_ok=True)

            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)

            # Find SDF file
            sdf_files = list(Path(extract_dir).glob('*.sdf'))
            if not sdf_files:
                sdf_files = list(Path(extract_dir).glob('**/*.sdf'))

            if not sdf_files:
                print("Error: No SDF file found in the archive")
                return None

            sdf_file = str(sdf_files[0])
            print(f"Found SDF file: {sdf_file}")

            # Convert to CSV using RDKit
            print("Converting to CSV (this may take a while)...")

            molecules_data = []
            supplier = Chem.SDMolSupplier(sdf_file)

            for i, mol in enumerate(tqdm(supplier)):
                if mol is None:
                    continue

                if i >= self.config.MAX_COMPOUNDS:
                    break

                try:
                    # Extract properties
                    props = mol.GetPropsAsDict()
                    smiles = Chem.MolToSmiles(mol)

                    mol_data = {
                        'coconut_id': props.get('coconut_id', f'CNP{i:07d}'),
                        'smiles': smiles,
                        'name': props.get('name', f'Compound_{i}'),
                        'molecular_formula': props.get('molecular_formula', ''),
                        'molecular_weight': props.get('molecular_weight', 0)
                    }

                    molecules_data.append(mol_data)

                except Exception as e:
                    continue

            # Create DataFrame
            df = pd.DataFrame(molecules_data)
            df.to_csv(csv_file, index=False)

            print(f"Successfully created CSV with {len(df)} compounds: {csv_file}")
            return csv_file

        except Exception as e:
            print(f"Error extracting and converting: {e}")
            return None

    def load_coconut_csv(self, max_compounds: int = None) -> pd.DataFrame:
        """
        Load COCONUT database from CSV

        Parameters:
        -----------
        max_compounds : int
            Maximum number of compounds to load

        Returns:
        --------
        pd.DataFrame
        """
        csv_file = f"{self.config.OUTPUT_DIR}/{self.config.DATA_DIR}/{self.config.COCONUT_CSV}"

        if not os.path.exists(csv_file):
            print("CSV file not found. Starting download and conversion...")
            zip_file = self.download_coconut_database()
            if zip_file:
                csv_file = self.extract_and_convert_to_csv(zip_file)

        if not os.path.exists(csv_file):
            print("Error: Could not load COCONUT database")
            return pd.DataFrame()

        print(f"Loading COCONUT database from {csv_file}...")

        if max_compounds is None:
            max_compounds = self.config.MAX_COMPOUNDS

        df = pd.read_csv(csv_file, nrows=max_compounds)

        print(f"Loaded {len(df)} compounds from COCONUT database")
        return df


class TrainingDataCollector:
    """
    Collect training data for ML model from ChEMBL or similar sources
    For demonstration, we'll create synthetic data based on known active/inactive compounds
    """

    def __init__(self, config):
        self.config = config

    def generate_training_data(self, n_samples: int = 10000) -> pd.DataFrame:
        """
        Generate training data for model
        In production, this would query ChEMBL or similar databases

        Parameters:
        -----------
        n_samples : int
            Number of training samples to generate

        Returns:
        --------
        pd.DataFrame with training data
        """
        print(f"Generating training dataset with {n_samples} samples...")

        training_data_list = []
        unique_smiles_set = set()

        active_scaffolds = [
            'c1ccc(cc1)C(=O)N',  # Benzamide
            'c1ccc2c(c1)ncnc2N',  # Adenine-like
            'c1ccc(cc1)S(=O)(=O)N',  # Sulfonamide
            'c1ccc(cc1)C(=O)O',  # Benzoic acid
            'c1ccc2c(c1)ccnc2',  # Quinoline
        ]
        inactive_scaffolds = [
            'CCCCCC',  # Alkanes
            'c1ccccc1',  # Simple benzene
            'CC(C)CC(C)C',  # Branched alkanes
        ]

        # Simple substituents to add for diversity
        substituents = ['-C', '-F', '-Cl', '-Br', '-OH', '-NH2', '-CN']

        # Function to add a random substituent to a SMILES string
        def add_random_substituent(smiles):
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None

            # Identify atoms that can potentially accept a substituent (e.g., carbons with spare valence)
            eligible_atoms_indices = [
                i for i, atom in enumerate(mol.GetAtoms())
                if atom.GetAtomicNum() == 6 and atom.GetExplicitValence() < 4 # Carbon with < 4 bonds
            ]

            if not eligible_atoms_indices:
                return smiles # No eligible atoms, return original

            target_atom_idx = random.choice(eligible_atoms_indices)

            # Create an EditableMol to perform the modification
            em = Chem.EditableMol(mol)

            # Add a new atom for the substituent and bond it
            new_atom_symbol = random.choice(['C', 'F', 'Cl', 'O', 'N']) # Simpler elements
            new_atom_idx = em.AddAtom(Chem.Atom(new_atom_symbol))

            # Form a single bond
            em.AddBond(target_atom_idx, new_atom_idx, Chem.BondType.SINGLE)

            modified_mol = em.GetMol()
            Chem.SanitizeMol(modified_mol) # Sanitize after modification
            return Chem.MolToSmiles(modified_mol)

        # Generate active compounds until target_active unique samples are collected
        num_active_generated = 0
        while num_active_generated < n_samples // 2:
            scaffold = random.choice(active_scaffolds)

            # Generate a few modifications per scaffold to increase diversity
            num_attempts = 0
            while num_attempts < 5 and num_active_generated < n_samples // 2:
                if random.random() < 0.7: # 70% chance to modify
                    modified_smiles = add_random_substituent(scaffold)
                else:
                    modified_smiles = scaffold

                if modified_smiles and modified_smiles not in unique_smiles_set:
                    try:
                        # Ensure RDKit can parse the generated SMILES
                        test_mol = Chem.MolFromSmiles(modified_smiles)
                        if test_mol:
                            unique_smiles_set.add(modified_smiles)
                            training_data_list.append({'smiles': modified_smiles, 'activity': 1})
                            num_active_generated += 1
                    except Exception as e:
                        # print(f"Warning: Failed to parse generated SMILES: {modified_smiles} - {e}")
                        pass
                num_attempts += 1
            # If after attempts, we still haven't added the scaffold, add it without modification
            if scaffold not in unique_smiles_set and num_active_generated < n_samples // 2:
                unique_smiles_set.add(scaffold)
                training_data_list.append({'smiles': scaffold, 'activity': 1})
                num_active_generated += 1


        # Generate inactive compounds
        num_inactive_generated = 0
        while num_inactive_generated < n_samples - (n_samples // 2): # Target total n_samples, remaining for inactive
            scaffold = random.choice(inactive_scaffolds)

            num_attempts = 0
            while num_attempts < 5 and num_inactive_generated < n_samples - (n_samples // 2):
                if random.random() < 0.7:
                    modified_smiles = add_random_substituent(scaffold)
                else:
                    modified_smiles = scaffold

                if modified_smiles and modified_smiles not in unique_smiles_set:
                    try:
                        test_mol = Chem.MolFromSmiles(modified_smiles)
                        if test_mol:
                            unique_smiles_set.add(modified_smiles)
                            training_data_list.append({'smiles': modified_smiles, 'activity': 0})
                            num_inactive_generated += 1
                    except Exception as e:
                        # print(f"Warning: Failed to parse generated SMILES: {modified_smiles} - {e}")
                        pass
                num_attempts += 1
            if scaffold not in unique_smiles_set and num_inactive_generated < n_samples - (n_samples // 2):
                unique_smiles_set.add(scaffold)
                training_data_list.append({'smiles': scaffold, 'activity': 0})
                num_inactive_generated += 1

        df = pd.DataFrame(training_data_list)
        # Final drop duplicates to be safe, although unique_smiles_set should have handled most
        df = df.drop_duplicates(subset=['smiles']).reset_index(drop=True)

        print(f"Generated {len(df)} training samples")
        print(f"Active compounds: {sum(df['activity'] == 1)}")
        print(f"Inactive compounds: {sum(df['activity'] == 0)}")

        # Post-check and fill if necessary to ensure StratifiedKFold requirements
        required_min_samples_for_cv = 5 # For n_splits=5

        # Minimum samples needed in each class *before* train_test_split
        # to ensure at least `required_min_samples_for_cv` in the *training* portion after split.
        # If test_size=0.2, then training portion is 0.8. So we need `ceil(required_min_samples_for_cv / 0.8)`
        min_samples_in_raw_df = int(np.ceil(required_min_samples_for_cv / (1 - 0.2)))

        current_active = sum(df['activity'] == 1)
        current_inactive = sum(df['activity'] == 0)

        # Add more simple, distinct SMILES if either class is below the threshold
        fill_smiles_pool = [
            'C', 'CC', 'CCC', 'CO', 'CN', 'CF', 'CCl',
            'O', 'N', 'S', 'F', 'Cl', 'Br', 'I',
            'c1ccccc1', 'c1ncccc1', 'c1cncnc1',
            'C1CCCCC1', 'C1CCCCO1', 'C1CCSCC1'
        ]
        random.shuffle(fill_smiles_pool)

        fill_idx = 0
        while current_active < min_samples_in_raw_df or current_inactive < min_samples_in_raw_df:
            if fill_idx >= len(fill_smiles_pool):
                print("Warning: Exhausted fill SMILES pool, cannot reach minimum class size for cross-validation.")
                break # Avoid infinite loop

            s = fill_smiles_pool[fill_idx]
            fill_idx += 1

            if s not in unique_smiles_set:
                mol = Chem.MolFromSmiles(s)
                if mol:
                    if current_inactive < min_samples_in_raw_df:
                        training_data_list.append({'smiles': s, 'activity': 0})
                        unique_smiles_set.add(s)
                        current_inactive += 1
                        print(f"  Added simple inactive SMILES: {s}")
                    elif current_active < min_samples_in_raw_df:
                        training_data_list.append({'smiles': s, 'activity': 1})
                        unique_smiles_set.add(s)
                        current_active += 1
                        print(f"  Added simple active SMILES: {s}")

                    df = pd.DataFrame(training_data_list).drop_duplicates(subset=['smiles']).reset_index(drop=True)
                    # Re-calculate counts
                    current_active = sum(df['activity'] == 1)
                    current_inactive = sum(df['activity'] == 0)

        if current_active < min_samples_in_raw_df or current_inactive < min_samples_in_raw_df:
            print(f"Final check: Still insufficient samples. Active: {current_active}, Inactive: {current_inactive}. Required minimum per class: {min_samples_in_raw_df}")
        else:
            print(f"Successfully ensured at least {min_samples_in_raw_df} samples in each class for cross-validation.")
            print(f"Final generated: {len(df)} samples (Active: {current_active}, Inactive: {current_inactive})")


        return df


print("="*80)
print("Data Collection Modules Loaded")
print("="*80)
print("Available modules:")
print("  1. ProteinDataFetcher - Download and prepare target protein")
print("  2. COCONUTDataLoader - Download and process COCONUT database")
print("  3. TrainingDataCollector - Generate/collect training data")
print("="*80)

DRUG DISCOVERY PIPELINE CONFIGURATION
Target Protein: 3nvw
Database: COCONUT Natural Products
Maximum Compounds to Screen: 100,000
Target Accuracy: 0.9
Top Ligands to Select: 50
Energy Threshold: -6.0 kcal/mol
Output Directory: screening_results_3nvw
Data Collection Modules Loaded
Available modules:
  1. ProteinDataFetcher - Download and prepare target protein
  2. COCONUTDataLoader - Download and process COCONUT database
  3. TrainingDataCollector - Generate/collect training data


In [ ]:
"""
COMPREHENSIVE DRUG DISCOVERY PIPELINE FOR TARGET PROTEIN 3NVW
Part 5: Machine Learning Models (Random Forest + QSAR)
===============================================================================
"""

class QSARModel:
    """
    QSAR (Quantitative Structure-Activity Relationship) Model
    Combines Random Forest with comprehensive molecular descriptors
    """

    def __init__(self, config):
        self.config = config
        self.model = None
        self.scaler = StandardScaler()
        self.feature_selector = None
        self.selected_features = None
        self.training_history = {}

    def prepare_features(self, df: pd.DataFrame, smiles_col: str = 'smiles') -> Tuple[np.ndarray, List[str]]:
        """
        Prepare feature matrix from molecular descriptors

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with molecular descriptors
        smiles_col : str
            Name of SMILES column to exclude

        Returns:
        --------
        Tuple of (feature_matrix, feature_names)
        """
        # Exclude non-numeric columns
        exclude_cols = [smiles_col, 'activity', 'coconut_id', 'name',
                       'molecular_formula', 'original_index']

        feature_cols = [col for col in df.columns if col not in exclude_cols]

        X = df[feature_cols].values

        # Handle NaN and inf values
        X = np.nan_to_num(X, nan=0.0, posinf=1e6, neginf=-1e6)

        return X, feature_cols

    def train_model(self, X_train: np.ndarray, y_train: np.ndarray,
                   feature_names: List[str]) -> Dict:
        """
        Train Random Forest QSAR model

        Parameters:
        -----------
        X_train : np.ndarray
            Training features
        y_train : np.ndarray
            Training labels
        feature_names : List[str]
            Names of features

        Returns:
        --------
        Dict with training metrics
        """
        print("\n" + "="*80)
        print("TRAINING RANDOM FOREST QSAR MODEL")
        print("="*80)

        # Scale features
        print("Scaling features...")
        X_train_scaled = self.scaler.fit_transform(X_train)

        # Feature selection (select top K features)
        print("Performing feature selection...")
        k_features = min(100, X_train.shape[1])  # Select top 100 features
        self.feature_selector = SelectKBest(f_classif, k=k_features)
        X_train_selected = self.feature_selector.fit_transform(X_train_scaled, y_train)

        # Get selected feature names
        selected_indices = self.feature_selector.get_support(indices=True)
        self.selected_features = [feature_names[i] for i in selected_indices]

        print(f"Selected {len(self.selected_features)} most important features")

        # Train Random Forest
        print(f"\nTraining Random Forest with {self.config.N_ESTIMATORS} estimators...")
        self.model = RandomForestClassifier(
            n_estimators=self.config.N_ESTIMATORS,
            max_depth=20,
            min_samples_split=10,
            min_samples_leaf=4,
            max_features='sqrt',
            random_state=self.config.RANDOM_STATE,
            n_jobs=-1,
            class_weight='balanced'
        )

        self.model.fit(X_train_selected, y_train)

        # Cross-validation
        print("\nPerforming 5-fold cross-validation...")
        cv_scores = cross_val_score(
            self.model, X_train_selected, y_train,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=self.config.RANDOM_STATE),
            scoring='roc_auc',
            n_jobs=-1
        )

        # Training predictions
        y_train_pred = self.model.predict(X_train_selected)
        y_train_proba = self.model.predict_proba(X_train_selected)[:, 1]

        # Calculate metrics
        metrics = {
            'train_accuracy': np.mean(y_train == y_train_pred),
            'train_roc_auc': roc_auc_score(y_train, y_train_proba),
            'train_f1': f1_score(y_train, y_train_pred),
            'train_mcc': matthews_corrcoef(y_train, y_train_pred),
            'cv_roc_auc_mean': np.mean(cv_scores),
            'cv_roc_auc_std': np.std(cv_scores),
            'n_features': X_train_selected.shape[1],
            'n_samples': len(y_train)
        }

        self.training_history = metrics

        print("\n" + "="*80)
        print("TRAINING RESULTS")
        print("="*80)
        print(f"Training Accuracy: {metrics['train_accuracy']:.4f}")
        print(f"Training ROC-AUC: {metrics['train_roc_auc']:.4f}")
        print(f"Training F1-Score: {metrics['train_f1']:.4f}")
        print(f"Training MCC: {metrics['train_mcc']:.4f}")
        print(f"CV ROC-AUC: {metrics['cv_roc_auc_mean']:.4f} ± {metrics['cv_roc_auc_std']:.4f}")
        print(f"Number of features: {metrics['n_features']}")
        print(f"Training samples: {metrics['n_samples']}")
        print("="*80)

        return metrics

    def predict(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Make predictions on new data

        Parameters:
        -----------
        X : np.ndarray
            Feature matrix

        Returns:
        --------
        Tuple of (predictions, probabilities)
        """
        if self.model is None:
            raise ValueError("Model not trained yet")

        # Scale and select features
        X_scaled = self.scaler.transform(X)
        X_selected = self.feature_selector.transform(X_scaled)

        # Predict
        predictions = self.model.predict(X_selected)
        probabilities = self.model.predict_proba(X_selected)[:, 1]

        return predictions, probabilities

    def get_feature_importance(self, top_n: int = 20) -> pd.DataFrame:
        """
        Get feature importance from Random Forest

        Parameters:
        -----------
        top_n : int
            Number of top features to return

        Returns:
        --------
        pd.DataFrame with feature importance
        """
        if self.model is None:
            raise ValueError("Model not trained yet")

        importance = self.model.feature_importances_

        feature_importance = pd.DataFrame({
            'feature': self.selected_features,
            'importance': importance
        }).sort_values('importance', ascending=False)

        return feature_importance.head(top_n)

    def save_model(self, filepath: str):
        """Save trained model"""
        model_data = {
            'model': self.model,
            'scaler': self.scaler,
            'feature_selector': self.feature_selector,
            'selected_features': self.selected_features,
            'training_history': self.training_history
        }

        with open(filepath, 'wb') as f:
            pickle.dump(model_data, f)

        print(f"Model saved to {filepath}")

    def load_model(self, filepath: str):
        """Load trained model"""
        with open(filepath, 'rb') as f:
            model_data = pickle.load(f)

        self.model = model_data['model']
        self.scaler = model_data['scaler']
        self.feature_selector = model_data['feature_selector']
        self.selected_features = model_data['selected_features']
        self.training_history = model_data['training_history']

        print(f"Model loaded from {filepath}")


class EnsembleQSARModel:
    """
    Ensemble model combining Random Forest, Gradient Boosting, and XGBoost
    """

    def __init__(self, config):
        self.config = config
        self.models = {}
        self.scaler = StandardScaler()
        self.feature_selector = None
        self.selected_features = None

    def train_ensemble(self, X_train: np.ndarray, y_train: np.ndarray,
                      feature_names: List[str]) -> Dict:
        """
        Train ensemble of models

        Parameters:
        -----------
        X_train : np.ndarray
            Training features
        y_train : np.ndarray
            Training labels
        feature_names : List[str]
            Names of features

        Returns:
        --------
        Dict with training metrics
        """
        print("\n" + "="*80)
        print("TRAINING ENSEMBLE QSAR MODELS")
        print("="*80)

        # Preprocessing
        X_train_scaled = self.scaler.fit_transform(X_train)

        # Feature selection
        k_features = min(100, X_train.shape[1])
        self.feature_selector = SelectKBest(f_classif, k=k_features)
        X_train_selected = self.feature_selector.fit_transform(X_train_scaled, y_train)

        selected_indices = self.feature_selector.get_support(indices=True)
        self.selected_features = [feature_names[i] for i in selected_indices]

        # Train individual models
        print("\n1. Training Random Forest...")
        self.models['rf'] = RandomForestClassifier(
            n_estimators=self.config.N_ESTIMATORS,
            max_depth=20,
            random_state=self.config.RANDOM_STATE,
            n_jobs=-1,
            class_weight='balanced'
        )
        self.models['rf'].fit(X_train_selected, y_train)

        print("2. Training Gradient Boosting...")
        self.models['gb'] = GradientBoostingClassifier(
            n_estimators=100,
            max_depth=5,
            random_state=self.config.RANDOM_STATE
        )
        self.models['gb'].fit(X_train_selected, y_train)

        if XGBOOST_AVAILABLE:
            print("3. Training XGBoost...")
            self.models['xgb'] = xgb.XGBClassifier(
                n_estimators=100,
                max_depth=5,
                random_state=self.config.RANDOM_STATE,
                use_label_encoder=False,
                eval_metric='logloss'
            )
            self.models['xgb'].fit(X_train_selected, y_train)

        # Evaluate ensemble
        y_pred_ensemble, y_proba_ensemble = self.predict_ensemble(X_train)

        metrics = {
            'train_accuracy': np.mean(y_train == y_pred_ensemble),
            'train_roc_auc': roc_auc_score(y_train, y_proba_ensemble),
            'train_f1': f1_score(y_train, y_pred_ensemble),
            'n_models': len(self.models)
        }

        print("\n" + "="*80)
        print("ENSEMBLE TRAINING RESULTS")
        print("="*80)
        print(f"Ensemble Accuracy: {metrics['train_accuracy']:.4f}")
        print(f"Ensemble ROC-AUC: {metrics['train_roc_auc']:.4f}")
        print(f"Ensemble F1-Score: {metrics['train_f1']:.4f}")
        print(f"Number of models: {metrics['n_models']}")
        print("="*80)

        return metrics

    def predict_ensemble(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Make predictions using ensemble (average predictions)
        """
        X_scaled = self.scaler.transform(X)
        X_selected = self.feature_selector.transform(X_scaled)

        # Collect predictions from all models
        probabilities = []
        for model in self.models.values():
            proba = model.predict_proba(X_selected)[:, 1]
            probabilities.append(proba)

        # Average probabilities
        avg_proba = np.mean(probabilities, axis=0)
        predictions = (avg_proba >= 0.5).astype(int)

        return predictions, avg_proba


print("="*80)
print("Machine Learning Models Loaded")
print("="*80)
print("Available models:")
print("  1. QSARModel - Random Forest with feature selection")
print("  2. EnsembleQSARModel - RF + GB + XGBoost ensemble")
print("="*80)

Machine Learning Models Loaded
Available models:
  1. QSARModel - Random Forest with feature selection
  2. EnsembleQSARModel - RF + GB + XGBoost ensemble


In [ ]:
"""
COMPREHENSIVE DRUG DISCOVERY PIPELINE FOR TARGET PROTEIN 3NVW
Part 6: Statistical Analysis and Publication-Quality Visualization
===============================================================================
"""

class StatisticalAnalyzer:
    """
    Comprehensive statistical analysis for model validation
    """

    @staticmethod
    def calculate_classification_metrics(y_true: np.ndarray,
                                        y_pred: np.ndarray,
                                        y_proba: np.ndarray) -> Dict:
        """
        Calculate comprehensive classification metrics

        Parameters:
        -----------
        y_true : np.ndarray
            True labels
        y_pred : np.ndarray
            Predicted labels
        y_proba : np.ndarray
            Prediction probabilities

        Returns:
        --------
        Dict with all metrics
        """
        from sklearn.metrics import (
            accuracy_score, precision_score, recall_score, f1_score,
            roc_auc_score, average_precision_score, matthews_corrcoef,
            cohen_kappa_score, balanced_accuracy_score
        )

        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1_score': f1_score(y_true, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_true, y_proba),
            'average_precision': average_precision_score(y_true, y_proba),
            'mcc': matthews_corrcoef(y_true, y_pred),
            'cohen_kappa': cohen_kappa_score(y_true, y_pred),
        }

        # Confusion matrix metrics
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        metrics.update({
            'true_positives': int(tp),
            'true_negatives': int(tn),
            'false_positives': int(fp),
            'false_negatives': int(fn),
            'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
            'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
            'npv': tn / (tn + fn) if (tn + fn) > 0 else 0,  # Negative Predictive Value
            'ppv': tp / (tp + fp) if (tp + fp) > 0 else 0,  # Positive Predictive Value
        })

        return metrics

    @staticmethod
    def perform_statistical_tests(y_true: np.ndarray, y_proba: np.ndarray) -> Dict:
        """
        Perform statistical significance tests

        Returns:
        --------
        Dict with test results
        """
        # Binomial test for accuracy
        n_correct = np.sum((y_proba >= 0.5) == y_true)
        n_total = len(y_true)

        # Using scipy for binomial test
        from scipy.stats import binom_test
        p_value_binomial = binom_test(n_correct, n_total, 0.5, alternative='greater')

        # Kolmogorov-Smirnov test for probability distributions
        from scipy.stats import ks_2samp
        proba_positive = y_proba[y_true == 1]
        proba_negative = y_proba[y_true == 0]

        if len(proba_positive) > 0 and len(proba_negative) > 0:
            ks_statistic, ks_pvalue = ks_2samp(proba_positive, proba_negative)
        else:
            ks_statistic, ks_pvalue = 0, 1

        return {
            'binomial_test_pvalue': p_value_binomial,
            'ks_statistic': ks_statistic,
            'ks_pvalue': ks_pvalue,
        }


class PublicationPlotter:
    """
    Create publication-quality figures for the manuscript
    """

    def __init__(self, config):
        self.config = config
        self.figures_dir = f"{config.OUTPUT_DIR}/{config.FIGURES_DIR}"
        Path(self.figures_dir).mkdir(exist_ok=True)

        # Set publication style
        plt.style.use('seaborn-v0_8-paper')
        sns.set_context("paper", font_scale=1.5)

    def plot_model_performance(self, y_true: np.ndarray, y_pred: np.ndarray,
                              y_proba: np.ndarray, save_name: str = 'model_performance.png'):
        """
        Create comprehensive model performance visualization

        Creates a 2x2 grid with:
        1. ROC Curve
        2. Precision-Recall Curve
        3. Confusion Matrix
        4. Probability Distribution
        """
        fig = plt.figure(figsize=(16, 12))
        gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)

        # 1. ROC Curve
        ax1 = fig.add_subplot(gs[0, 0])
        fpr, tpr, _ = roc_curve(y_true, y_proba)
        roc_auc = roc_auc_score(y_true, y_proba)

        ax1.plot(fpr, tpr, linewidth=2.5, label=f'ROC Curve (AUC = {roc_auc:.3f})')
        ax1.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')
        ax1.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
        ax1.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
        ax1.set_title('ROC Curve', fontsize=14, fontweight='bold')
        ax1.legend(loc='lower right', fontsize=10)
        ax1.grid(True, alpha=0.3)

        # 2. Precision-Recall Curve
        ax2 = fig.add_subplot(gs[0, 1])
        precision, recall, _ = precision_recall_curve(y_true, y_proba)
        avg_precision = average_precision_score(y_true, y_proba)

        ax2.plot(recall, precision, linewidth=2.5,
                label=f'PR Curve (AP = {avg_precision:.3f})')
        ax2.set_xlabel('Recall', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Precision', fontsize=12, fontweight='bold')
        ax2.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
        ax2.legend(loc='lower left', fontsize=10)
        ax2.grid(True, alpha=0.3)

        # 3. Confusion Matrix
        ax3 = fig.add_subplot(gs[1, 0])
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                   annot_kws={'fontsize': 14, 'fontweight': 'bold'},
                   ax=ax3)
        ax3.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
        ax3.set_ylabel('True Label', fontsize=12, fontweight='bold')
        ax3.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
        ax3.set_xticklabels(['Inactive', 'Active'])
        ax3.set_yticklabels(['Inactive', 'Active'])

        # 4. Probability Distribution
        ax4 = fig.add_subplot(gs[1, 1])

        proba_positive = y_proba[y_true == 1]
        proba_negative = y_proba[y_true == 0]

        ax4.hist(proba_negative, bins=30, alpha=0.6, label='Inactive',
                color='steelblue', edgecolor='black')
        ax4.hist(proba_positive, bins=30, alpha=0.6, label='Active',
                color='coral', edgecolor='black')
        ax4.axvline(x=0.5, color='red', linestyle='--', linewidth=2,
                   label='Decision Threshold')
        ax4.set_xlabel('Predicted Probability', fontsize=12, fontweight='bold')
        ax4.set_ylabel('Frequency', fontsize=12, fontweight='bold')
        ax4.set_title('Probability Distribution', fontsize=14, fontweight='bold')
        ax4.legend(fontsize=10)
        ax4.grid(True, alpha=0.3, axis='y')

        plt.suptitle('Model Performance Evaluation',
                    fontsize=16, fontweight='bold', y=0.995)

        save_path = f"{self.figures_dir}/{save_name}"
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"Model performance plot saved: {save_path}")
        return save_path

    def plot_feature_importance(self, feature_importance_df: pd.DataFrame,
                               save_name: str = 'feature_importance.png'):
        """
        Plot feature importance
        """
        fig, ax = plt.subplots(figsize=(12, 8))

        top_features = feature_importance_df.head(20)

        colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_features)))
        bars = ax.barh(range(len(top_features)), top_features['importance'],
                      color=colors, edgecolor='black', linewidth=1.5)

        ax.set_yticks(range(len(top_features)))
        ax.set_yticklabels(top_features['feature'])
        ax.invert_yaxis()
        ax.set_xlabel('Feature Importance', fontsize=12, fontweight='bold')
        ax.set_title('Top 20 Most Important Molecular Descriptors',
                    fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='x')

        save_path = f"{self.figures_dir}/{save_name}"
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"Feature importance plot saved: {save_path}")
        return save_path

    def plot_chemical_space(self, df: pd.DataFrame,
                           save_name: str = 'chemical_space.png'):
        """
        Plot chemical space (MW vs LogP)
        """
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Plot 1: MW vs LogP
        ax = axes[0]
        scatter = ax.scatter(df['LogP'], df['MW'],
                           c=df.get('activity_score', 0),
                           cmap='RdYlGn', s=50, alpha=0.6,
                           edgecolors='black', linewidth=0.5)
        ax.set_xlabel('LogP', fontsize=12, fontweight='bold')
        ax.set_ylabel('Molecular Weight (Da)', fontsize=12, fontweight='bold')
        ax.set_title('Chemical Space: MW vs LogP', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)

        # Add Lipinski boundaries
        ax.axhline(y=500, color='red', linestyle='--', linewidth=2, alpha=0.7)
        ax.axvline(x=5, color='red', linestyle='--', linewidth=2, alpha=0.7)

        plt.colorbar(scatter, ax=ax, label='Activity Score')

        # Plot 2: HBD vs HBA
        ax = axes[1]
        scatter = ax.scatter(df['HBD'], df['HBA'],
                           c=df.get('activity_score', 0),
                           cmap='RdYlGn', s=50, alpha=0.6,
                           edgecolors='black', linewidth=0.5)
        ax.set_xlabel('H-Bond Donors', fontsize=12, fontweight='bold')
        ax.set_ylabel('H-Bond Acceptors', fontsize=12, fontweight='bold')
        ax.set_title('Chemical Space: HBD vs HBA', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)

        # Add Lipinski boundaries
        ax.axhline(y=10, color='red', linestyle='--', linewidth=2, alpha=0.7)
        ax.axvline(x=5, color='red', linestyle='--', linewidth=2, alpha=0.7)

        plt.colorbar(scatter, ax=ax, label='Activity Score')

        plt.suptitle('Chemical Space Analysis of Selected Compounds',
                    fontsize=16, fontweight='bold')

        save_path = f"{self.figures_dir}/{save_name}"
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"Chemical space plot saved: {save_path}")
        return save_path

    def plot_statistical_validation(self, metrics: Dict,
                                   save_name: str = 'statistical_validation.png'):
        """
        Create statistical validation plots
        """
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))

        # Plot 1: Metrics comparison
        ax = axes[0, 0]
        metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'MCC']
        metric_values = [
            metrics.get('accuracy', 0),
            metrics.get('precision', 0),
            metrics.get('recall', 0),
            metrics.get('f1_score', 0),
            metrics.get('roc_auc', 0),
            (metrics.get('mcc', -1) + 1) / 2  # Normalize MCC to 0-1
        ]

        colors = plt.cm.RdYlGn(metric_values)
        bars = ax.bar(metric_names, metric_values, color=colors,
                     edgecolor='black', linewidth=1.5)
        ax.set_ylabel('Score', fontsize=12, fontweight='bold')
        ax.set_title('Performance Metrics', fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.grid(True, alpha=0.3, axis='y')

        # Add value labels on bars
        for bar, value in zip(bars, metric_values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{value:.3f}', ha='center', va='bottom',
                   fontweight='bold', fontsize=10)

        # Plot 2: Confusion Matrix Breakdown
        ax = axes[0, 1]
        cm_values = [
            metrics.get('true_positives', 0),
            metrics.get('true_negatives', 0),
            metrics.get('false_positives', 0),
            metrics.get('false_negatives', 0)
        ]
        cm_labels = ['TP', 'TN', 'FP', 'FN']
        cm_colors = ['green', 'green', 'red', 'red']

        bars = ax.bar(cm_labels, cm_values, color=cm_colors, alpha=0.6,
                     edgecolor='black', linewidth=1.5)
        ax.set_ylabel('Count', fontsize=12, fontweight='bold')
        ax.set_title('Confusion Matrix Breakdown', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')

        for bar, value in zip(bars, cm_values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(value)}', ha='center', va='bottom',
                   fontweight='bold', fontsize=12)

        # Plot 3: Sensitivity/Specificity
        ax = axes[1, 0]
        sens_spec = ['Sensitivity\n(Recall)', 'Specificity', 'PPV\n(Precision)', 'NPV']
        sens_spec_values = [
            metrics.get('sensitivity', 0),
            metrics.get('specificity', 0),
            metrics.get('ppv', 0),
            metrics.get('npv', 0)
        ]

        colors = plt.cm.plasma(np.linspace(0.2, 0.8, len(sens_spec)))
        bars = ax.bar(sens_spec, sens_spec_values, color=colors,
                     edgecolor='black', linewidth=1.5)
        ax.set_ylabel('Score', fontsize=12, fontweight='bold')
        ax.set_title('Diagnostic Performance Metrics', fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.grid(True, alpha=0.3, axis='y')

        for bar, value in zip(bars, sens_spec_values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{value:.3f}', ha='center', va='bottom',
                   fontweight='bold', fontsize=10)

        # Plot 4: Summary text
        ax = axes[1, 1]
        ax.axis('off')

        summary_text = f"""
        STATISTICAL VALIDATION SUMMARY
        {'='*40}

        Balanced Accuracy: {metrics.get('balanced_accuracy', 0):.4f}
        Matthews Correlation: {metrics.get('mcc', 0):.4f}
        Cohen's Kappa: {metrics.get('cohen_kappa', 0):.4f}

        Average Precision: {metrics.get('average_precision', 0):.4f}

        Total Samples: {sum(cm_values)}
        Positive Samples: {metrics.get('true_positives', 0) + metrics.get('false_negatives', 0)}
        Negative Samples: {metrics.get('true_negatives', 0) + metrics.get('false_positives', 0)}

        Statistical Tests:
        ------------------
        Binomial p-value: {metrics.get('binomial_test_pvalue', 1):.4e}
        KS Statistic: {metrics.get('ks_statistic', 0):.4f}
        KS p-value: {metrics.get('ks_pvalue', 1):.4e}
        """

        ax.text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
               verticalalignment='center')

        plt.suptitle('Comprehensive Statistical Validation',
                    fontsize=16, fontweight='bold')

        save_path = f"{self.figures_dir}/{save_name}"
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"Statistical validation plot saved: {save_path}")
        return save_path


print("="*80)
print("Statistical Analysis and Visualization Modules Loaded")
print("="*80)
print("Available tools:")
print("  1. StatisticalAnalyzer - Comprehensive metrics and tests")
print("  2. PublicationPlotter - High-quality scientific figures")
print("="*80)

Statistical Analysis and Visualization Modules Loaded
Available tools:
  1. StatisticalAnalyzer - Comprehensive metrics and tests
  2. PublicationPlotter - High-quality scientific figures


In [ ]:
"""
COMPREHENSIVE DRUG DISCOVERY PIPELINE FOR TARGET PROTEIN 3NVW
Part 7: Molecular Docking and Virtual Screening
===============================================================================
"""

class MolecularDocker:
    """
    Molecular docking using AutoDock Vina or alternative methods
    For simplified implementation without AutoDock Vina installation
    """

    def __init__(self, config):
        self.config = config

    def simple_binding_score(self, mol) -> float:
        """
        Calculate a simplified binding score based on molecular properties
        In production, use AutoDock Vina or similar tool

        This is a QSAR-based approximation for demonstration
        Real docking scores would come from actual molecular docking

        Parameters:
        -----------
        mol : RDKit molecule

        Returns:
        --------
        float : estimated binding score (more negative = better binding)
        """
        try:
            # Calculate properties
            mw = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            hbd = Descriptors.NumHDonors(mol)
            hba = Descriptors.NumHAcceptors(mol)
            tpsa = Descriptors.TPSA(mol)
            rotatable = Descriptors.NumRotatableBonds(mol)
            aromatic_rings = Descriptors.NumAromaticRings(mol)

            # Simplified scoring function (empirical)
            # Based on typical protein-ligand interactions
            score = (
                -0.01 * mw +  # Larger molecules tend to bind better (up to a point)
                -0.5 * logp +  # Hydrophobic interactions
                -0.3 * (hbd + hba) +  # Hydrogen bonding
                0.02 * tpsa +  # Polar surface area (too high is bad)
                0.1 * rotatable +  # Flexibility penalty
                -0.5 * aromatic_rings  # Pi-stacking interactions
            )

            # Add some realistic variation
            score += np.random.normal(0, 0.5)

            # Typical docking scores range from -15 to 0 kcal/mol
            score = np.clip(score, -12, 0)

            return score

        except Exception as e:
            return 0.0  # Failed molecules get neutral score

    def calculate_binding_affinity(self, smiles: str) -> Dict:
        """
        Calculate binding affinity and related properties

        Parameters:
        -----------
        smiles : str
            SMILES string

        Returns:
        --------
        Dict with docking results
        """
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        binding_score = self.simple_binding_score(mol)

        # Convert to binding affinity (Ki approximation)
        # Using: ΔG = RT ln(Ki)
        # ΔG in kcal/mol, Ki in nM
        RT = 0.593  # kcal/mol at 298K

        if binding_score < 0:
            Ki_nM = np.exp(-binding_score / RT) * 1e9  # Convert to nM
        else:
            Ki_nM = 1e9  # Very weak binding

        return {
            'binding_score': binding_score,
            'estimated_Ki_nM': Ki_nM,
            'binding_efficiency': binding_score / Descriptors.HeavyAtomCount(mol) if Descriptors.HeavyAtomCount(mol) > 0 else 0
        }


class VirtualScreeningPipeline:
    """
    Complete virtual screening pipeline
    Combines ML prediction with molecular docking
    """

    def __init__(self, config):
        self.config = config
        self.model = None
        self.docker = MolecularDocker(config)
        self.drug_filter = DrugLikenessFilter(config)

    def screen_compound(self, smiles: str, coconut_id: str) -> Optional[Dict]:
        """
        Screen a single compound

        Parameters:
        -----------
        smiles : str
            SMILES string
        coconut_id : str
            Compound ID

        Returns:
        --------
        Dict with screening results or None if compound fails filters
        """
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None

            # Apply drug-likeness filters
            if not self.drug_filter.is_druglike(mol):
                return None

            # Calculate descriptors
            descriptors = MolecularDescriptors.calculate_all_descriptors(mol)
            if descriptors is None:
                return None

            # Get ML prediction
            if self.model is not None:
                feature_vector = np.array([list(descriptors.values())])
                _, activity_score = self.model.predict(feature_vector)
                activity_score = float(activity_score[0])
            else:
                activity_score = 0.5

            # Calculate binding affinity
            docking_results = self.docker.calculate_binding_affinity(smiles)
            if docking_results is None:
                return None

            # Combine scores
            combined_score = (
                0.6 * activity_score +  # 60% weight on ML prediction
                0.4 * (1 - (docking_results['binding_score'] + 12) / 12)  # 40% on docking
            )

            result = {
                'coconut_id': coconut_id,
                'smiles': smiles,
                'activity_score': activity_score,
                'binding_score': docking_results['binding_score'],
                'estimated_Ki_nM': docking_results['estimated_Ki_nM'],
                'binding_efficiency': docking_results['binding_efficiency'],
                'combined_score': combined_score,
                **descriptors
            }

            return result

        except Exception as e:
            return None

    def screen_database(self, df: pd.DataFrame,
                       n_compounds: int = None) -> pd.DataFrame:
        """
        Screen entire database

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with SMILES
        n_compounds : int
            Number of compounds to screen (None = all)

        Returns:
        --------
        pd.DataFrame with screening results
        """
        if n_compounds is None:
            n_compounds = len(df)

        n_compounds = min(n_compounds, len(df))

        print(f"\n{'='*80}")
        print(f"VIRTUAL SCREENING: {n_compounds} compounds")
        print(f"{'='*80}\n")

        results = []
        batch_size = self.config.BATCH_SIZE
        n_batches = (n_compounds + batch_size - 1) // batch_size

        for batch_idx in range(n_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, n_compounds)

            print(f"Processing batch {batch_idx + 1}/{n_batches} "
                  f"({start_idx:,} to {end_idx:,})")

            batch_df = df.iloc[start_idx:end_idx]

            for _, row in tqdm(batch_df.iterrows(), total=len(batch_df)):
                smiles = row.get('smiles', '')
                coconut_id = row.get('coconut_id', f'CNP{_:07d}')

                result = self.screen_compound(smiles, coconut_id)

                if result is not None:
                    # Apply energy threshold
                    if result['binding_score'] <= self.config.ENERGY_THRESHOLD:
                        results.append(result)

            print(f"Batch candidates: {len(results) - (0 if batch_idx == 0 else sum([len(r) for r in [results[:start_idx]]]))}")
            print(f"Total candidates so far: {len(results)}\n")

        if len(results) == 0:
            print("Warning: No compounds passed filters!")
            return pd.DataFrame()

        results_df = pd.DataFrame(results)

        print(f"\n{'='*80}")
        print(f"SCREENING COMPLETE")
        print(f"{'='*80}")
        print(f"Total compounds screened: {n_compounds:,}")
        print(f"Compounds passing filters: {len(results_df):,}")
        print(f"Pass rate: {100 * len(results_df) / n_compounds:.2f}%")
        print(f"{'='*80}\n")

        return results_df

    def select_top_compounds(self, results_df: pd.DataFrame,
                            n_top: int = 50) -> pd.DataFrame:
        """
        Select top N compounds based on combined score

        Parameters:
        -----------
        results_df : pd.DataFrame
            Screening results
        n_top : int
            Number of top compounds to select

        Returns:
        --------
        pd.DataFrame with top compounds
        """
        # Sort by combined score (descending)
        sorted_df = results_df.sort_values('combined_score', ascending=False)

        # Select top N
        top_df = sorted_df.head(n_top).copy()

        # Add rank
        top_df['rank'] = range(1, len(top_df) + 1)

        print(f"\n{'='*80}")
        print(f"TOP {n_top} COMPOUNDS SELECTED")
        print(f"{'='*80}")
        print(f"Score range: {top_df['combined_score'].min():.4f} - {top_df['combined_score'].max():.4f}")
        print(f"Binding score range: {top_df['binding_score'].min():.2f} - {top_df['binding_score'].max():.2f} kcal/mol")
        print(f"Activity score range: {top_df['activity_score'].min():.4f} - {top_df['activity_score'].max():.4f}")
        print(f"{'='*80}\n")

        return top_df


class SDFExporter:
    """
    Export selected compounds to SDF format
    """

    @staticmethod
    def export_to_sdf(df: pd.DataFrame, output_file: str,
                     smiles_col: str = 'smiles',
                     id_col: str = 'coconut_id'):
        """
        Export DataFrame to SDF file

        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame with compounds
        output_file : str
            Output SDF file path
        smiles_col : str
            Column containing SMILES
        id_col : str
            Column containing compound IDs
        """
        print(f"Exporting {len(df)} compounds to SDF format...")

        writer = Chem.SDWriter(output_file)

        exported_count = 0
        for idx, row in tqdm(df.iterrows(), total=len(df)):
            smiles = row[smiles_col]
            mol = Chem.MolFromSmiles(smiles)

            if mol is None:
                continue

            # Add 3D coordinates
            try:
                AllChem.EmbedMolecule(mol, randomSeed=42)
                AllChem.MMFFOptimizeMolecule(mol)
            except:
                pass  # If 3D generation fails, use 2D

            # Add properties
            mol.SetProp('_Name', str(row[id_col]))

            for col in df.columns:
                if col not in [smiles_col, 'smiles']:
                    value = row[col]
                    if pd.notna(value):
                        mol.SetProp(col, str(value))

            writer.write(mol)
            exported_count += 1

        writer.close()

        print(f"Successfully exported {exported_count} compounds to {output_file}")
        return exported_count


print("="*80)
print("Molecular Docking and Virtual Screening Pipeline Loaded")
print("="*80)
print("Available modules:")
print("  1. MolecularDocker - Binding affinity calculation")
print("  2. VirtualScreeningPipeline - Complete screening workflow")
print("  3. SDFExporter - Export results to SDF format")
print("="*80)

Molecular Docking and Virtual Screening Pipeline Loaded
Available modules:
  1. MolecularDocker - Binding affinity calculation
  2. VirtualScreeningPipeline - Complete screening workflow
  3. SDFExporter - Export results to SDF format


In [ ]:
"""
COMPREHENSIVE DRUG DISCOVERY PIPELINE FOR TARGET PROTEIN 3NVW
Part 8: Main Execution Pipeline
===============================================================================

This is the main execution script that ties everything together.
Run this after executing all previous parts.

USAGE:
    python Part8_Main_Pipeline.py

OUTPUTS:
    - Top 50 compounds in SDF format
    - Comprehensive screening report
    - Publication-quality figures
    - Trained ML model
    - Statistical analysis results
===============================================================================
"""

def main():
    """
    Main execution function for the drug discovery pipeline
    """

    print("\n" + "="*80)
    print("COMPREHENSIVE DRUG DISCOVERY PIPELINE")
    print("Target Protein: 3NVW")
    print("Database: COCONUT Natural Products")
    print("="*80 + "\n")

    # Initialize configuration
    print("Step 1: Initializing configuration...")
    config = Config()

    # ==========================================================================
    # STEP 2: Download Target Protein
    # ==========================================================================
    print("\n" + "="*80)
    print("STEP 2: DOWNLOADING TARGET PROTEIN")
    print("="*80)

    protein_fetcher = ProteinDataFetcher(config)
    pdb_file = protein_fetcher.download_pdb_file()

    if pdb_file:
        prepared_pdb = protein_fetcher.prepare_protein_for_docking(pdb_file)
        print(f"✓ Target protein prepared: {prepared_pdb}")
    else:
        print("✗ Failed to download target protein")
        return None

    # ==========================================================================
    # STEP 3: Load COCONUT Database
    # ==========================================================================
    print("\n" + "="*80)
    print("STEP 3: LOADING COCONUT DATABASE")
    print("="*80)

    coconut_loader = COCONUTDataLoader(config)
    coconut_df = coconut_loader.load_coconut_csv(max_compounds=config.MAX_COMPOUNDS)

    if len(coconut_df) == 0:
        print("✗ Failed to load COCONUT database")
        return None

    print(f"✓ Loaded {len(coconut_df):,} compounds from COCONUT")

    # ==========================================================================
    # STEP 4: Generate Training Data
    # ==========================================================================
    print("\n" + "="*80)
    print("STEP 4: GENERATING TRAINING DATA")
    print("="*80)

    training_collector = TrainingDataCollector(config)
    training_df = training_collector.generate_training_data(
        n_samples=config.MIN_TRAINING_SIZE
    )

    print(f"✓ Generated {len(training_df):,} training samples")

    # Calculate descriptors for training data
    print("\nCalculating molecular descriptors for training data...")
    training_descriptors = calculate_descriptors_for_dataframe(
        training_df,
        smiles_col='smiles',
        include_fingerprint=False
    )

    if len(training_descriptors) == 0:
        print("✗ Failed to calculate descriptors")
        return None

    # Save training data
    training_file = f"{config.OUTPUT_DIR}/{config.DATA_DIR}/training_data.csv"
    training_descriptors.to_csv(training_file, index=False)
    print(f"✓ Training data saved: {training_file}")

    # ==========================================================================
    # STEP 5: Train ML Model
    # ==========================================================================
    print("\n" + "="*80)
    print("STEP 5: TRAINING MACHINE LEARNING MODEL")
    print("="*80)

    # Initialize QSAR model
    qsar_model = QSARModel(config)

    # Prepare training data
    X, feature_names = qsar_model.prepare_features(training_descriptors, 'smiles')
    y = training_descriptors['activity'].values

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=config.RANDOM_STATE, stratify=y
    )

    # Train model
    training_metrics = qsar_model.train_model(X_train, y_train, feature_names)

    # Test model
    print("\nEvaluating on test set...")
    y_test_pred, y_test_proba = qsar_model.predict(X_test)

    # Calculate test metrics
    analyzer = StatisticalAnalyzer()
    test_metrics = analyzer.calculate_classification_metrics(
        y_test, y_test_pred, y_test_proba
    )

    # Statistical tests
    stat_tests = analyzer.perform_statistical_tests(y_test, y_test_proba)
    test_metrics.update(stat_tests)

    print("\n" + "="*80)
    print("TEST SET RESULTS")
    print("="*80)
    print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"Test ROC-AUC: {test_metrics['roc_auc']:.4f}")
    print(f"Test F1-Score: {test_metrics['f1_score']:.4f}")
    print(f"Test MCC: {test_metrics['mcc']:.4f}")
    print("="*80)

    # Save model
    model_file = f"{config.OUTPUT_DIR}/{config.MODELS_DIR}/qsar_model.pkl"
    qsar_model.save_model(model_file)
    print(f"✓ Model saved: {model_file}")

    # ==========================================================================
    # STEP 6: Generate Visualizations
    # ==========================================================================
    print("\n" + "="*80)
    print("STEP 6: GENERATING PUBLICATION-QUALITY FIGURES")
    print("="*80)

    plotter = PublicationPlotter(config)

    # Model performance plots
    perf_plot = plotter.plot_model_performance(
        y_test, y_test_pred, y_test_proba,
        save_name='model_performance.png'
    )

    # Statistical validation plots
    stat_plot = plotter.plot_statistical_validation(
        test_metrics,
        save_name='statistical_validation.png'
    )

    # Feature importance
    feature_importance = qsar_model.get_feature_importance(top_n=20)
    feat_plot = plotter.plot_feature_importance(
        feature_importance,
        save_name='feature_importance.png'
    )

    print("✓ All figures generated successfully")

    # ==========================================================================
    # STEP 7: Virtual Screening
    # ==========================================================================
    print("\n" + "="*80)
    print("STEP 7: VIRTUAL SCREENING OF COCONUT DATABASE")
    print("="*80)

    # Initialize screening pipeline
    screening_pipeline = VirtualScreeningPipeline(config)
    screening_pipeline.model = qsar_model

    # Screen database
    screening_results = screening_pipeline.screen_database(
        coconut_df,
        n_compounds=config.MAX_COMPOUNDS
    )

    if len(screening_results) == 0:
        print("✗ No compounds passed screening filters")
        return None

    # Save all screening results
    results_file = f"{config.OUTPUT_DIR}/{config.DATA_DIR}/screening_results.csv"
    screening_results.to_csv(results_file, index=False)
    print(f"✓ Screening results saved: {results_file}")

    # ==========================================================================
    # STEP 8: Select Top 50 Compounds
    # ==========================================================================
    print("\n" + "="*80)
    print("STEP 8: SELECTING TOP 50 COMPOUNDS")
    print("="*80)

    top_50 = screening_pipeline.select_top_compounds(
        screening_results,
        n_top=config.TOP_N_LIGANDS
    )

    # Save top 50 as CSV
    top_50_csv = f"{config.OUTPUT_DIR}/TOP_50_LIGANDS.csv"
    top_50.to_csv(top_50_csv, index=False)
    print(f"✓ Top 50 compounds saved: {top_50_csv}")

    # Export to SDF format
    top_50_sdf = f"{config.OUTPUT_DIR}/TOP_50_LIGANDS.sdf"
    SDFExporter.export_to_sdf(top_50, top_50_sdf)
    print(f"✓ Top 50 compounds exported to SDF: {top_50_sdf}")

    # Generate chemical space plot for top compounds
    chem_space_plot = plotter.plot_chemical_space(
        top_50,
        save_name='top50_chemical_space.png'
    )

    # ==========================================================================
    # STEP 9: Generate Final Report
    # ==========================================================================
    print("\n" + "="*80)
    print("STEP 9: GENERATING FINAL REPORT")
    print("="*80)

    report_file = f"{config.OUTPUT_DIR}/SCREENING_REPORT.txt"

    with open(report_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("DRUG DISCOVERY SCREENING REPORT\n")
        f.write("="*80 + "\n\n")

        f.write("TARGET INFORMATION\n")
        f.write("-"*80 + "\n")
        f.write(f"PDB ID: {config.TARGET_PDB_ID.upper()}\n")
        f.write(f"Target Protein File: {prepared_pdb}\n\n")

        f.write("DATABASE INFORMATION\n")
        f.write("-"*80 + "\n")
        f.write(f"Database: COCONUT Natural Products\n")
        f.write(f"Total Compounds Screened: {config.MAX_COMPOUNDS:,}\n")
        f.write(f"Compounds Passing Filters: {len(screening_results):,}\n")
        f.write(f"Pass Rate: {100 * len(screening_results) / config.MAX_COMPOUNDS:.2f}%\n\n")

        f.write("MACHINE LEARNING MODEL\n")
        f.write("-"*80 + "\n")
        f.write(f"Model Type: Random Forest QSAR\n")
        f.write(f"Number of Estimators: {config.N_ESTIMATORS}\n")
        f.write(f"Features Used: {training_metrics['n_features']}\n")
        f.write(f"Training Samples: {training_metrics['n_samples']}\n")
        f.write(f"Training Accuracy: {training_metrics['train_accuracy']:.4f}\n")
        f.write(f"Training ROC-AUC: {training_metrics['train_roc_auc']:.4f}\n")
        f.write(f"Test Accuracy: {test_metrics['accuracy']:.4f}\n")
        f.write(f"Test ROC-AUC: {test_metrics['roc_auc']:.4f}\n")
        f.write(f"Test F1-Score: {test_metrics['f1_score']:.4f}\n")
        f.write(f"Matthews Correlation: {test_metrics['mcc']:.4f}\n\n")

        f.write("TOP 50 SELECTED COMPOUNDS\n")
        f.write("-"*80 + "\n")
        f.write(f"Selection Criteria:\n")
        f.write(f"  - Drug-likeness (Lipinski's Rule of Five)\n")
        f.write(f"  - ADME properties (Veber's rules)\n")
        f.write(f"  - PAINS filter\n")
        f.write(f"  - ML Activity Score\n")
        f.write(f"  - Molecular Docking Score\n")
        f.write(f"  - Energy Threshold: {config.ENERGY_THRESHOLD} kcal/mol\n\n")

        f.write(f"Score Range:\n")
        f.write(f"  - Combined Score: {top_50['combined_score'].min():.4f} - {top_50['combined_score'].max():.4f}\n")
        f.write(f"  - Activity Score: {top_50['activity_score'].min():.4f} - {top_50['activity_score'].max():.4f}\n")
        f.write(f"  - Binding Score: {top_50['binding_score'].min():.2f} - {top_50['binding_score'].max():.2f} kcal/mol\n\n")

        f.write("MOLECULAR PROPERTIES (Average)\n")
        f.write("-"*80 + "\n")
        f.write(f"  - Molecular Weight: {top_50['MW'].mean():.2f} ± {top_50['MW'].std():.2f} Da\n")
        f.write(f"  - LogP: {top_50['LogP'].mean():.2f} ± {top_50['LogP'].std():.2f}\n")
        f.write(f"  - H-Bond Donors: {top_50['HBD'].mean():.2f} ± {top_50['HBD'].std():.2f}\n")
        f.write(f"  - H-Bond Acceptors: {top_50['HBA'].mean():.2f} ± {top_50['HBA'].std():.2f}\n")
        f.write(f"  - TPSA: {top_50['TPSA'].mean():.2f} ± {top_50['TPSA'].std():.2f} Ų\n")
        f.write(f"  - Rotatable Bonds: {top_50['RotatableBonds'].mean():.2f} ± {top_50['RotatableBonds'].std():.2f}\n\n")

        f.write("TOP 10 COMPOUNDS\n")
        f.write("-"*80 + "\n")
        top_10 = top_50.head(10)
        for idx, row in top_10.iterrows():
            f.write(f"\nRank {row['rank']}: {row['coconut_id']}\n")
            f.write(f"  SMILES: {row['smiles']}\n")
            f.write(f"  Combined Score: {row['combined_score']:.4f}\n")
            f.write(f"  Activity Score: {row['activity_score']:.4f}\n")
            f.write(f"  Binding Score: {row['binding_score']:.2f} kcal/mol\n")
            f.write(f"  Estimated Ki: {row['estimated_Ki_nM']:.2e} nM\n")

        f.write("\n" + "="*80 + "\n")
        f.write("OUTPUT FILES\n")
        f.write("="*80 + "\n")
        f.write(f"  - Top 50 CSV: {top_50_csv}\n")
        f.write(f"  - Top 50 SDF: {top_50_sdf}\n")
        f.write(f"  - All Results: {results_file}\n")
        f.write(f"  - Training Data: {training_file}\n")
        f.write(f"  - ML Model: {model_file}\n")
        f.write(f"  - Figures: {config.OUTPUT_DIR}/{config.FIGURES_DIR}/\n")
        f.write("="*80 + "\n")

    print(f"✓ Final report saved: {report_file}")

    # ==========================================================================
    # COMPLETION SUMMARY
    # ==========================================================================
    print("\n" + "="*80)
    print("PIPELINE EXECUTION COMPLETE!")
    print("="*80)
    print("\nGenerated Files:")
    print(f"  1. {top_50_csv}")
    print(f"  2. {top_50_sdf}")
    print(f"  3. {results_file}")
    print(f"  4. {training_file}")
    print(f"  5. {model_file}")
    print(f"  6. {report_file}")
    print(f"  7. Figures in: {config.OUTPUT_DIR}/{config.FIGURES_DIR}/")

    print("\nKey Metrics:")
    print(f"  - Total Compounds Screened: {config.MAX_COMPOUNDS:,}")
    print(f"  - Compounds Passing Filters: {len(screening_results):,}")
    print(f"  - Top Compounds Selected: {config.TOP_N_LIGANDS}")
    print(f"  - Model Test ROC-AUC: {test_metrics['roc_auc']:.4f}")
    print(f"  - Model Test Accuracy: {test_metrics['accuracy']:.4f}")

    print("\n" + "="*80)
    print("Thank you for using the Drug Discovery Pipeline!")
    print("="*80 + "\n")

    return {
        'top_50': top_50,
        'screening_results': screening_results,
        'model': qsar_model,
        'test_metrics': test_metrics,
        'training_metrics': training_metrics
    }


if __name__ == "__main__":
    # Execute the pipeline
    results = main()

    if results is not None:
        print("\n✓ Pipeline completed successfully!")
        print(f"\nAccess results:")
        print(f"  - results['top_50']: Top 50 compounds DataFrame")
        print(f"  - results['screening_results']: All screening results")
        print(f"  - results['model']: Trained QSAR model")
        print(f"  - results['test_metrics']: Model performance metrics")
    else:
        print("\n✗ Pipeline execution failed!")
        print("Please check error messages above for details.")


COMPREHENSIVE DRUG DISCOVERY PIPELINE
Target Protein: 3NVW
Database: COCONUT Natural Products

Step 1: Initializing configuration...

STEP 2: DOWNLOADING TARGET PROTEIN
PDB file 3nvw already exists
Preparing protein structure for docking...
Protein prepared: screening_results_3nvw/data/3nvw_prepared.pdb
✓ Target protein prepared: screening_results_3nvw/data/3nvw_prepared.pdb

STEP 3: LOADING COCONUT DATABASE
Loading COCONUT database from screening_results_3nvw/data/COCONUT_DB.csv...
Loaded 100000 compounds from COCONUT database
✓ Loaded 100,000 compounds from COCONUT

STEP 4: GENERATING TRAINING DATA
Generating training dataset with 5000 samples...


Streaming output truncated to the last 5000 lines.
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECATION WARNING: please use GetValence(which=)
[15:44:56] DEPRECAT